# pymr 全流程演示

本 Notebook 展示使用 `pymr` 完成两样本孟德尔随机化（Two-Sample MR）的完整流程：

1. **一键分析** — `mr_pipeline()` 自动完成重叠检测→协调→MR分析→敏感性分析
2. **常量内存** — 仅加载重叠 SNP 数据，不加载全量 GWAS 数据到内存
3. **完整可视化** — 散点图、森林图、漏斗图、QQ 图等

In [1]:
exposure_data_path = '/mnt/disk2/dataset/mr/ebi-a-GCST90093110'
outcome_data_path = '/mnt/disk2/dataset/mr/ukb-e-250_CSA'

# max_snps: 按p-value选取暴露数据中最显著的N个SNP
# 增大此值可获得更多SNP，但会增加内存使用和运行时间
MAX_SNPS = 500_000

## 数据读取

In [11]:
from pymr import load_gwas


exposure_data =load_gwas(exposure_data_path)
exposure_data =load_gwas(outcome_data_path)

In [10]:
exposure_data

<xarray.Dataset> Size: 30GB
Dimensions:         (variant: 9812567)
Coordinates:
    alt             (variant) <U256 10GB ...
    chrom           (variant) int8 10MB ...
    id              (variant) <U256 10GB ...
    pos             (variant) int32 39MB ...
    ref             (variant) <U256 10GB ...
Dimensions without coordinates: variant
Data variables:
    eaf             (variant) float32 39MB ...
    effect_size     (variant) float32 39MB ...
    info_af         (variant) float32 39MB ...
    log_pvalue      (variant) float32 39MB ...
    standard_error  (variant) float32 39MB ...
Attributes:
    study_id:             ukb-e-250_CSA
    study_type:           Continuous
    total_variants:       9812567
    harmonised_variants:  9812567
    total_cases:          None
    total_controls:       None
    source_vcf:           /mnt/disk2/dataset/mr/ukb-e-250_CSA/ukb-e-250_CSA.v...
    format_fields:        ES,SE,LP,AF

## 数据调谐

## 1. 一键 MR 分析

使用 `mr_pipeline()` 自动完成：重叠检测 → 协调 → MR 分析 → 敏感性分析。

In [2]:
import time
from pymr import mr_pipeline

t0 = time.time()

result = mr_pipeline(
    exposure_data_path,
    outcome_data_path,
    max_snps=MAX_SNPS,
)

print(f"Pipeline completed in {time.time()-t0:.1f}s")
print(result.summary())

Pipeline completed in 56.1s
MR Pipeline Summary
Exposure: /mnt/disk2/dataset/mr/ebi-a-GCST90093110
  Total variants: 13,291,573
Outcome: /mnt/disk2/dataset/mr/ukb-e-250_CSA
  Total variants: 9,812,567

Overlap: 213,997 SNPs
Harmonised: 196,735 SNPs
Alignment rate: 100.0%

Main Analysis:
  IVW                 : b=  0.3573, SE=  0.4807, P=4.57e-01 
  MR Egger            : b= -0.0703, SE=  0.0811, P=3.86e-01 
  Weighted Median     : b=  0.0665, SE=  0.2909, P=8.19e-01 
  Simple Mode         : b= -2.8478, SE=  1.9980, P=1.54e-01 

Heterogeneity:
  IVW: Q_P=0.00e+00
  MR Egger: Q_P=1.00e+00

Pleiotropy (Egger): DETECTED (P=0.0000)
MR-PRESSO: Global P=0.5020, Outliers=49951

Weak IVs (F<10): 196735


## 2. 数据协调（Harmonisation）

确保暴露与结局数据中每个 SNP 的效应等位基因方向一致，使 β 值可比。

In [ ]:
## 2. 数据协调详情

data = result.harmonise.data
print(f"协调后 SNP 数量: {data['action'].value_counts().to_dict()}")

In [ ]:
data.head(10)

In [ ]:
data['action'].value_counts()

## 3. 主分析（Main MR Analysis）

使用四种 MR 方法估计因果效应。

In [ ]:
## 3. 主分析结果

mr_results = result.mr_results
snp_results = result.single_snp

pd.DataFrame([{
    'Method': r.method, 'NSNP': r.nsnp,
    'Beta': r.b, 'SE': r.se, 'P': r.pval,
    '95% CI': f'[{r.ci_low:.4f}, {r.ci_up:.4f}]',
} for r in mr_results])

In [ ]:
snp_results.nsmallest(10, 'pval')

## 4. 敏感性分析（Sensitivity Analysis）

评估主分析结果的稳健性，检测和校正潜在偏倚。

### 4.1 异质性检验

In [ ]:
### 4.1 异质性检验

for h in result.sensitivity['heterogeneity']:
    sig = 'YES' if h.Q_pval < 0.05 else 'NO'
    print(f"{h.method}: Q = {h.Q:.2f}, df = {h.Q_df:.0f}, P = {h.Q_pval:.2e}  [Heterogeneous: {sig}]")

### 4.2 多效性检验（Egger 截距）

In [ ]:
### 4.2 多效性检验（Egger 截距）

ei = result.sensitivity['egger_intercept']
print(f"Egger intercept = {ei.egger_intercept:.6f}")
print(f"SE = {ei.se:.6f}")
print(f"P = {ei.pval:.4f}")
print(f"\n定向水平多效性: {'检测到 (P < 0.05)' if ei.pval < 0.05 else '未检测到'}")

### 4.3 MR-PRESSO 全局检验

In [ ]:
### 4.3 MR-PRESSO 全局检验

pr = result.sensitivity['mr_presso']
if pr:
    print(f"Global Q = {pr.global_Q:.2f}, df = {pr.global_Q_df:.0f}")
    print(f"Global P = {pr.global_pval:.4f}")
    print(f"Outliers detected: {pr.n_outliers}")
    if pr.n_outliers > 0:
        print(f"Top outlier SNPs: {pr.outlier_snps[:10]}")
    if pr.corrected_b is not None:
        print(f"Corrected estimate: Beta = {pr.corrected_b:.6f}, P = {pr.corrected_pval:.4f}")
else:
    print("MR-PRESSO not computed.")

### 4.4 工具变量强度（F 统计量）

In [ ]:
### 4.4 工具变量强度（F 统计量）

f_stats = result.sensitivity.get('f_statistics', [])
f_vals = np.array([f.F for f in f_stats if np.isfinite(f.F) and f.F > 0])

print(f"工具变量数量: {len(f_vals):,}")
if len(f_vals) > 0:
    print(f"F 统计量: min = {f_vals.min():.2f}, median = {np.median(f_vals):.2f}, mean = {f_vals.mean():.2f}")
    print(f"弱工具 (F < 10): {(f_vals < 10).sum():,} / {len(f_vals):,} ({(f_vals < 10).mean():.1%})")
else:
    print("F statistics not computed (sample_size not provided).")

### 4.5 留一法分析（Leave-One-Out）

In [ ]:
### 4.5 留一法分析（Leave-One-Out）

from pymr import leave_one_out

# 选取最显著的 20 个 SNP 进行留一法
top_snps = snp_results.nsmallest(20, 'pval')['SNP']
data_top = data[data['SNP'].isin(top_snps)].reset_index(drop=True)

loo_results = leave_one_out(data_top, method='ivw')

loo_df = pd.DataFrame([{
    'Excluded SNP': r.SNP,
    'Beta': r.b,
    'SE': r.se,
    'P': r.pval,
} for r in loo_results])

ivw_result = [r for r in mr_results if r.method == 'IVW'][0]
print(f"Full IVW estimate: Beta = {ivw_result.b:.6f}")
loo_df

### 4.6 一键综合敏感性分析

In [ ]:
### 4.6 一键综合敏感性分析

sensitivity = result.sensitivity

print("=" * 50)
print("  敏感性分析综合报告")
print("=" * 50)
print(f"  工具变量总数: {sensitivity['total_ivs']:,}")
print(f"  弱工具 (F<10): {sensitivity['weak_iv_count']:,}")
print(f"  F 统计量范围: [{sensitivity['min_F']:.2f}, {sensitivity['mean_F']:.2f}]")

het = sensitivity['heterogeneity']
for h in het:
    print(f"  异质性 ({h.method}): Q_P = {h.Q_pval:.2e}")

ei = sensitivity['egger_intercept']
if ei:
    print(f"  Egger 截距: {ei.egger_intercept:.6f}, P = {ei.pval:.4f}")

pr = sensitivity['mr_presso']
if pr:
    print(f"  MR-PRESSO: Global P = {pr.global_pval:.4f}, Outliers = {pr.n_outliers}")

## 5. 可视化

生成所有标准 MR 图表。

### 5.1 散点图（Scatter Plot）

In [ ]:
from pymr import scatter_plot

p = scatter_plot(data, mr_results,
                title='MR Scatter Plot\nExposure: GCST90093110 → Outcome: ukb-e-250_CSA')
p.save('/tmp/mr_scatter.png', dpi=150, width=10, height=7, verbose=False)
p

### 5.2 森林图（Forest Plot）

In [ ]:
from pymr import forest_plot

p = forest_plot(mr_results, title='MR Method Comparison')
p.save('/tmp/mr_forest.png', dpi=150, width=10, height=5, verbose=False)
p

### 5.3 漏斗图（Funnel Plot）

In [ ]:
from pymr import funnel_plot

p = funnel_plot(data, title='Funnel Plot (Symmetry Check)')
p.save('/tmp/mr_funnel.png', dpi=150, width=10, height=7, verbose=False)
p

### 5.4 QQ 图

In [ ]:
from pymr import qq_plot

p = qq_plot(data, title='QQ Plot of Wald Ratio P-values')
p.save('/tmp/mr_qq.png', dpi=150, width=10, height=7, verbose=False)
p

### 5.5 单 SNP 森林图

In [ ]:
from pymr import single_snp_forest

p = single_snp_forest(snp_results, ivw_result, top_n=30,
                      title='Single SNP Wald Ratios (Top 30)')
p.save('/tmp/mr_single_snp.png', dpi=150, width=10, height=10, verbose=False)
p

### 5.6 异质性森林图

In [ ]:
from pymr import heterogeneity_forest

p = heterogeneity_forest(data, top_n=30,
                       title='Top 30 Heterogeneity Contributors')
p.save('/tmp/mr_het_forest.png', dpi=150, width=10, height=10, verbose=False)
p

### 5.7 留一法图

In [ ]:
from pymr import leave_one_out_plot

p = leave_one_out_plot(loo_results, ivw_result, top_n=20,
                      title='Leave-One-Out Analysis')
p.save('/tmp/mr_loo.png', dpi=150, width=10, height=8, verbose=False)
p

### 5.8 方法方向一致性图

In [ ]:
from pymr import direction_consistency_plot

p = direction_consistency_plot(mr_results,
                            title='Method Direction Consistency')
p.save('/tmp/mr_direction.png', dpi=150, width=10, height=6, verbose=False)
p

## 6. 结果解读

综合以上分析进行最终因果推断解读。

In [ ]:
print("=" * 60)
print("  最终因果推断报告")
print("=" * 60)

# 主分析结果
print("\n[主分析结果]")
for r in mr_results:
    stars = '***' if r.pval < 0.001 else ('**' if r.pval < 0.01 else ('*' if r.pval < 0.05 else ''))
    print(f"  {r.method:20s}: Beta = {r.b:8.4f}, P = {r.pval:.2e} {stars}")

# 敏感性检验
print("\n[敏感性检验]")
het = sensitivity['heterogeneity']
het_ivw = het[0]
het_egger = het[1]
ei = sensitivity['egger_intercept']
pr = sensitivity.get('mr_presso')
print(f"  异质性 (IVW):     P = {het_ivw.Q_pval:.2e} {'[显著]' if het_ivw.Q_pval < 0.05 else ''}")
print(f"  异质性 (Egger):   P = {het_egger.Q_pval:.2e} {'[显著]' if het_egger.Q_pval < 0.05 else ''}")
if ei:
    print(f"  Egger 截距:       P = {ei.pval:.4f} {'[显著 - 存在多效性]' if ei.pval < 0.05 else '[不显著]'}")
if pr:
    print(f"  MR-PRESSO 全局:   P = {pr.global_pval:.4f}")
print(f"  弱工具 (F<10):    {sensitivity['weak_iv_count']:,} / {sensitivity['total_ivs']:,}")

# 综合判断
print("\n[综合判断]")
main_sig = all(r.pval < 0.05 for r in mr_results[:3])
pleiotropy = ei.pval < 0.05 if ei else False
hetero = het_ivw.Q_pval < 0.05

if main_sig and not pleiotropy:
    print("  结论: 支持因果关系的证据较强")
    print("  理由: 多种方法方向一致 + 无显著多效性")
elif main_sig and pleiotropy:
    print("  结论: 需谨慎解读")
    print("  理由: 方法方向一致但存在定向多效性")
    print("  建议: 以 MR-Egger 结果为准，或剔除多效性 SNP 后重新分析")
else:
    print("  结论: 因果关系证据不足")
    print("  理由: 方法间结果不一致")